# VNFood Hierarchical Training — Metrics CSV/Excel

Notebook đã được chia cell để chạy trên Kaggle. File metric sẽ được lưu vào `/kaggle/working` gồm `epoch_metrics.csv`, `final_metrics_summary.csv`, và `metrics_summary.xlsx`.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

from sklearn.metrics import classification_report, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score, matthews_corrcoef, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')


In [ ]:
# ============================================================
# END-TO-END HIERARCHICAL TEST EVALUATION
# GROUP_EFFB0 -> SUB_EFFB2 -> final food class
# ============================================================


def evaluate_hierarchical_end_to_end(group_model, group_ckpt, sub_models, test_dir):
    sorted_class_names = [name for name, _ in sorted(class_to_idx.items(), key=lambda item: item[1])]
    label_ids = list(range(len(sorted_class_names)))

    image_paths = []
    for cls_name in sorted_class_names:
        cls_dir = Path(test_dir) / cls_name
        if cls_dir.exists():
            image_paths.extend((cls_name, p) for p in cls_dir.glob('*') if p.is_file())

    if not image_paths:
        raise ValueError(f'No test images found in {test_dir}')

    y_true = []
    y_pred = []
    y_scores = []
    pred_rows = []

    for true_name, image_path in tqdm(image_paths, desc='Test HIERARCHICAL_EFFB0_EFFB2', leave=False):
        pred_group, group_conf, pred_name, sub_conf = predict_hierarchical(
            str(image_path), group_model, group_ckpt, sub_models
        )
        true_idx = class_to_idx[true_name]
        pred_idx = class_to_idx.get(pred_name, -1)
        score_vec = np.zeros(len(label_ids), dtype=np.float32)
        if pred_idx >= 0:
            score_vec[pred_idx] = float(group_conf) * float(sub_conf)

        y_true.append(true_idx)
        y_pred.append(pred_idx)
        y_scores.append(score_vec)
        pred_rows.append({
            'image_path': str(image_path),
            'true_class': true_name,
            'predicted_group': pred_group,
            'group_confidence': group_conf,
            'predicted_class': pred_name,
            'sub_confidence': sub_conf,
            'correct': pred_idx == true_idx,
        })

    model_name = 'HIERARCHICAL_EFFB0_EFFB2'
    diagnostics_dir = OUTPUT_DIR / 'hierarchical_end_to_end'
    diagnostics_dir.mkdir(exist_ok=True)

    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=label_ids, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=label_ids, average='weighted', zero_division=0
    )

    pred_csv_path = diagnostics_dir / f'{model_name}_predictions.csv'
    report_csv_path = diagnostics_dir / f'{model_name}_classification_report.csv'
    cm_csv_path = diagnostics_dir / f'{model_name}_confusion_matrix.csv'
    cm_png_path = diagnostics_dir / f'{model_name}_confusion_matrix.png'
    roc_csv_path = diagnostics_dir / f'{model_name}_roc_curve.csv'
    roc_png_path = diagnostics_dir / f'{model_name}_roc_curve.png'

    pd.DataFrame(pred_rows).to_csv(pred_csv_path, index=False)
    report_df = pd.DataFrame(classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=sorted_class_names,
        zero_division=0,
        output_dict=True
    )).T
    report_df.to_csv(report_csv_path)

    cm_labels = label_ids + ([-1] if -1 in y_pred else [])
    cm_names = sorted_class_names + (['unknown'] if -1 in y_pred else [])
    cm = confusion_matrix(y_true, y_pred, labels=cm_labels)
    cm_df = pd.DataFrame(cm, index=cm_names, columns=cm_names)
    cm_df.to_csv(cm_csv_path)

    y_score = np.asarray(y_scores)
    y_true_bin = label_binarize(y_true, classes=label_ids)
    roc_rows = []
    roc_auc_values = {}
    if y_score.ndim == 2 and y_score.shape[1] == len(sorted_class_names) and len(sorted_class_names) > 1:
        try:
            roc_auc_values['roc_auc_macro_ovr'] = roc_auc_score(
                y_true, y_score, labels=label_ids, multi_class='ovr', average='macro'
            )
            roc_auc_values['roc_auc_weighted_ovr'] = roc_auc_score(
                y_true, y_score, labels=label_ids, multi_class='ovr', average='weighted'
            )
            roc_auc_values['roc_auc_micro_ovr'] = roc_auc_score(
                y_true_bin, y_score, average='micro'
            )
        except ValueError as exc:
            print(f'ROC AUC skipped for {model_name}: {exc}')

        plt.figure(figsize=(9, 7))
        for class_idx, class_name in enumerate(sorted_class_names):
            if len(np.unique(y_true_bin[:, class_idx])) < 2:
                continue
            fpr, tpr, _ = roc_curve(y_true_bin[:, class_idx], y_score[:, class_idx])
            class_auc = auc(fpr, tpr)
            for x, y in zip(fpr, tpr):
                roc_rows.append({'class_name': class_name, 'fpr': x, 'tpr': y, 'auc': class_auc})
            plt.plot(fpr, tpr, linewidth=1, alpha=0.55, label=f'{class_name} AUC={class_auc:.3f}')

        fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_score.ravel())
        micro_auc = auc(fpr_micro, tpr_micro)
        for x, y in zip(fpr_micro, tpr_micro):
            roc_rows.append({'class_name': 'micro_average', 'fpr': x, 'tpr': y, 'auc': micro_auc})
        plt.plot(fpr_micro, tpr_micro, color='black', linewidth=2.5, label=f'micro AUC={micro_auc:.3f}')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'One-vs-Rest ROC - {model_name}')
        plt.legend(loc='lower right', fontsize=7, ncol=2)
        plt.tight_layout()
        plt.savefig(roc_png_path, dpi=160, bbox_inches='tight')
        plt.show()
        pd.DataFrame(roc_rows).to_csv(roc_csv_path, index=False)

    print(f'\n📊 END-TO-END TEST METRICS — {model_name}')
    print(f'Accuracy           : {acc:.4f}')
    print(f'Balanced Accuracy  : {balanced_acc:.4f}')
    print(f'MCC                : {mcc:.4f}')
    print(f'Macro Precision    : {macro_p:.4f}')
    print(f'Macro Recall       : {macro_r:.4f}')
    print(f'Macro F1           : {macro_f1:.4f}')
    print(f'Weighted Precision : {weighted_p:.4f}')
    print(f'Weighted Recall    : {weighted_r:.4f}')
    print(f'Weighted F1        : {weighted_f1:.4f}')
    print(f'Predictions CSV    : {pred_csv_path}')
    print(f'Classification CSV : {report_csv_path}')
    print(f'Confusion CSV      : {cm_csv_path}')
    print(f'Confusion PNG      : {cm_png_path}')
    print(f'ROC curve CSV      : {roc_csv_path}')
    print(f'ROC curve PNG      : {roc_png_path}')
    print(f'\nCLASSIFICATION REPORT — {model_name}')
    print(classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=sorted_class_names,
        zero_division=0
    ))
    print(f'\nCONFUSION MATRIX — {model_name}')
    display(cm_df)

    fig_size = max(8, min(24, 0.45 * max(len(cm_names), 1)))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_names).plot(
        ax=ax,
        cmap='Blues',
        xticks_rotation=90,
        colorbar=False,
        values_format='d'
    )
    ax.set_title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.savefig(cm_png_path, dpi=160, bbox_inches='tight')
    plt.show()

    metric = {
        'model_name': model_name,
        'split': 'test',
        'loss': np.nan,
        'accuracy': acc,
        'balanced_accuracy': balanced_acc,
        'mcc': mcc,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f1,
        'num_samples': len(y_true),
        'predictions_csv': str(pred_csv_path),
        'classification_report_csv': str(report_csv_path),
        'confusion_matrix_csv': str(cm_csv_path),
        'confusion_matrix_png': str(cm_png_path),
        'roc_curve_csv': str(roc_csv_path),
        'roc_curve_png': str(roc_png_path),
        **roc_auc_values,
    }

    if 'test_metrics_records' in globals():
        test_metrics_records[:] = [r for r in test_metrics_records if r.get('model_name') != model_name]
        test_metrics_records.append(metric)
        updated_test_metrics_df = pd.DataFrame(test_metrics_records)
    else:
        updated_test_metrics_df = pd.DataFrame([metric])

    updated_test_metrics_df.to_csv(OUTPUT_DIR / 'test_metrics_summary.csv', index=False)
    updated_test_metrics_df.to_excel(OUTPUT_DIR / 'test_metrics_summary.xlsx', index=False)
    display(updated_test_metrics_df)

    return metric, pd.DataFrame(pred_rows), cm_df


if all(name in globals() for name in ['group_model_inf', 'group_ckpt_inf', 'sub_models_inf', 'TEST_DIR']):
    hierarchical_e2e_metric, hierarchical_e2e_predictions, hierarchical_e2e_cm = evaluate_hierarchical_end_to_end(
        group_model_inf,
        group_ckpt_inf,
        sub_models_inf,
        TEST_DIR
    )
else:
    print('End-to-end metric helper loaded. Run after the model-loading cell to compute HIERARCHICAL_EFFB0_EFFB2 metrics.')


## 1. SETUP & SEED

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. SETUP & SEED


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 2. CONFIGURATION & GROUP DEFINITIONS

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. CONFIGURATION & GROUP DEFINITIONS


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

# PATHS
DATA_DIR = Path('/kaggle/input/datasets/meowluvmatcha/vnfood-30-100/vnfood_combined_dataset')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

# METRIC OUTPUT FILES
EPOCH_METRICS_CSV = OUTPUT_DIR / 'epoch_metrics.csv'
FINAL_METRICS_CSV = OUTPUT_DIR / 'final_metrics_summary.csv'
METRICS_EXCEL_PATH = OUTPUT_DIR / 'metrics_summary.xlsx'

# In-memory logs; these will be saved repeatedly during training.
epoch_metric_rows = []
final_metric_rows = []


def save_metrics_tables(output_dir=OUTPUT_DIR):
    """Save epoch-level and final model metrics to CSV + Excel.

    Files created in /kaggle/working:
      - epoch_metrics.csv
      - final_metrics_summary.csv
      - metrics_summary.xlsx with two sheets: epoch_metrics, final_metrics
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    epoch_df = pd.DataFrame(epoch_metric_rows)
    final_df = pd.DataFrame(final_metric_rows)

    if len(epoch_df) > 0:
        epoch_df.to_csv(EPOCH_METRICS_CSV, index=False)

    if len(final_df) > 0:
        final_df.to_csv(FINAL_METRICS_CSV, index=False)

    with pd.ExcelWriter(METRICS_EXCEL_PATH, engine='openpyxl') as writer:
        if len(epoch_df) > 0:
            epoch_df.to_excel(writer, sheet_name='epoch_metrics', index=False)
        else:
            pd.DataFrame({'message': ['No epoch metrics yet']}).to_excel(
                writer, sheet_name='epoch_metrics', index=False
            )

        if len(final_df) > 0:
            final_df.to_excel(writer, sheet_name='final_metrics', index=False)
        else:
            pd.DataFrame({'message': ['No final metrics yet']}).to_excel(
                writer, sheet_name='final_metrics', index=False
            )

    print(f'📁 Metrics saved:')
    if len(epoch_df) > 0:
        print(f'   - {EPOCH_METRICS_CSV}')
    if len(final_df) > 0:
        print(f'   - {FINAL_METRICS_CSV}')
    print(f'   - {METRICS_EXCEL_PATH}')

    return epoch_df, final_df

# TRAINING HYPERPARAMS
BATCH_SIZE    = 32
NUM_WORKERS   = 2
DROPOUT       = 0.3
WEIGHT_DECAY  = 1e-4
LABEL_SMOOTHING = 0.1

# Phase epochs for Group Classifier (EfficientNet-B0)
GROUP_PHASE1_EPOCHS = 5
GROUP_PHASE1_LR     = 1e-3
GROUP_PHASE2_EPOCHS = 10
GROUP_PHASE2_LR     = 1e-4
GROUP_PHASE3_EPOCHS = 15
GROUP_PHASE3_LR     = 5e-5

# Phase epochs for Sub-class Classifiers (EfficientNet-B2)
SUB_PHASE1_EPOCHS = 5
SUB_PHASE1_LR     = 1e-3
SUB_PHASE2_EPOCHS = 10
SUB_PHASE2_LR     = 1e-4
SUB_PHASE3_EPOCHS = 15
SUB_PHASE3_LR     = 5e-5

# Image sizes
GROUP_IMG_SIZE = 224   # EfficientNet-B0
SUB_IMG_SIZE   = 260   # EfficientNet-B2

print('Config loaded ✓')


In [ ]:
# ─── GROUP DEFINITIONS ────────────────────────────────────────────────────────
# Note: banh-canh appears in BOTH Group 1 and Group 2 (shared class)
# Some classes appear in multiple groups (e.g. ca-muoi-xoi, bo-kho)

GROUP_CLASSES = OrderedDict({
    'BANH': [
        'banh-bao', 'banh-beo', 'banh-bo', 'banh-bot-loc', 'banh-can',
        'banh-canh', 'banh-chung', 'banh-cong', 'banh-cuon', 'banh-da-cua',
        'banh-da-lon', 'banh-duc', 'banh-gai', 'banh-giay', 'banh-gio',
        'banh-hoi', 'banh-khot', 'banh-la', 'banh-mi', 'banh-mi-chao',
        'banh-pia', 'banh-tai-heo', 'banh-tet', 'banh-tieu',
        'banh-tom-ho-tay', 'banh-trang-nuong', 'banh-troi-nuoc',
        'banh-trung-thu', 'banh-u', 'banh-xeo', 'cao-lau',
    ],
    'BUN_PHO': [
        'pho', 'bun-bo-hue', 'bun-cha', 'bun-cha-ca',
        'bun-dau-mam-tom', 'bun-mam', 'bun-rieu', 'bun-thit-nuong',
        'hu-tieu', 'mi-quang', 'mi-xao-gion', 'nui-xao', 'nam-pia',
        'banh-canh',
    ],
    'COM': [
        'com-chay-cha-bong', 'com-chien', 'com-ga-xoi-mo',
        'com-lam', 'com-rang-dua-bo', 'com-tam',
    ],
    'MON_KHO_NUONG': [
        'bo-kho', 'bo-la-lot', 'bo-luc-lac', 'bo-ne', 'bo-nuong-la-lot',
        'ca-kho-to', 'ca-loc-nuong', 'ca-muoi-xoi', 'ca-sot-ca-chua',
        'ga-chien-nuoc-mam', 'kho-muc-nuong', 'kho-quet', 'lap-xuong',
        'luon-xao-xa-ot', 'muc-nhoi-thit', 'rau-muong-xao', 'thit-kho-tau',
    ],
    'CANH_CHAO': [
        'canh-bi-do', 'canh-chua', 'canh-cua', 'canh-kho-hoa',
        'canh-khoai-tim', 'ca-ri-ga', 'chao-long', 'chao-vit',
        'sup-cua', 'bo-kho', 'luon-om-chuoi-dau',
    ],
    'XOI': [
        'xoi-gac', 'xoi-nep-than', 'xoi-xeo',
    ],
    'GOI_CUON': [
        'goi-ca-chich', 'goi-cuon', 'nem-chua', 'nem-nuong-nha-trang',
        'cha-com', 'cha-lui',
    ],
    'DAC_BIET': [
        'baba-nau-chuoi-dau', 'ca-muoi-xoi', 'cha-ca-la-vong',
        'cua-hap-bia', 'cut-lon-xao-me', 'ga-hap-la-chanh', 'khau-nhuc',
        'mam-chung', 'mam-tep-chung-thit', 'oc-buou-hap', 'oc-huong-xao',
        'oc-len-xao-dua', 'tau-hu-nhoi-thit', 'tau-hu-non', 'thit-dong',
        'thit-trau-gac-bep', 'tiet-canh', 'trung-vit-lon',
    ],
})

NUM_GROUPS = len(GROUP_CLASSES)
GROUP_NAMES = list(GROUP_CLASSES.keys())
group_to_idx = {g: i for i, g in enumerate(GROUP_NAMES)}

# Build reverse map: class_name → group_name (use FIRST group if duplicated)
class_to_group = {}
for gname, cls_list in GROUP_CLASSES.items():
    for cls in cls_list:
        if cls not in class_to_group:
            class_to_group[cls] = gname

print(f'Number of groups: {NUM_GROUPS}')
for gname, cls_list in GROUP_CLASSES.items():
    print(f'  {gname:20s}: {len(cls_list)} classes')
print(f'\nTotal unique classes across all groups: {len(class_to_group)}')


## 3. EXPLORE & LOAD DATASET

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. EXPLORE & LOAD DATASET


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def find_split_dirs(root: Path):
    """Auto-detect train/val split"""
    for t in ['train', 'Train', 'training']:
        if (root / t).exists():
            train_dir = root / t
            for v in ['val', 'valid', 'validation', 'test']:
                if (root / v).exists():
                    return train_dir, root / v
            return train_dir, None
    return root, None


TRAIN_DIR, VAL_DIR = find_split_dirs(DATA_DIR)
TEST_DIR = DATA_DIR / "test"

print(f'Test dir  : {TEST_DIR}')
print(f'Train dir : {TRAIN_DIR}')
print(f'Val dir   : {VAL_DIR}')

# All 103 original classes
all_class_names = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
all_class_to_idx = {c: i for i, c in enumerate(all_class_names)}
print(f'Total classes found on disk: {len(all_class_names)}')

# Verify all group classes exist in dataset
missing = [c for c in class_to_group if c not in all_class_to_idx]
if missing:
    print(f'WARNING: classes in groups but not on disk: {missing}')
else:
    print('All group classes found on disk ✓')


## 4. TRANSFORMS

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. TRANSFORMS


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size + 20, img_size + 20)),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.RandomGrayscale(p=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    return train_tf, val_tf

print('Transforms factory defined ✓')


## 5. DATASET CLASSES

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. DATASET CLASSES


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

class FoodDataset(Dataset):
    """Generic dataset that loads images from class-name subdirectories."""
    def __init__(self, root_dir, class_to_idx, transform=None,
                 extensions=('.jpg', '.jpeg', '.png', '.webp')):
        self.root_dir     = Path(root_dir)
        self.class_to_idx = class_to_idx
        self.transform    = transform
        self.samples      = []

        for cls, idx in class_to_idx.items():
            cls_dir = self.root_dir / cls
            if not cls_dir.exists():
                continue
            for f in cls_dir.iterdir():
                if f.suffix.lower() in extensions:
                    self.samples.append((str(f), idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (260, 260))
        if self.transform:
            img = self.transform(img)
        return img, label


class GroupLabelDataset(Dataset):
    """Wraps a FoodDataset but returns GROUP labels instead of class labels."""
    def __init__(self, root_dir, class_names, class_to_group_map, group_to_idx_map,
                 transform=None, extensions=('.jpg', '.jpeg', '.png', '.webp')):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        for cls in class_names:
            if cls not in class_to_group_map:
                continue
            gname = class_to_group_map[cls]
            gidx  = group_to_idx_map[gname]
            cls_dir = self.root_dir / cls
            if not cls_dir.exists():
                continue
            for f in cls_dir.iterdir():
                if f.suffix.lower() in extensions:
                    self.samples.append((str(f), gidx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224))
        if self.transform:
            img = self.transform(img)
        return img, label

print('Dataset classes defined ✓')


## 6. MODEL BUILDERS & TRAINING UTILITIES

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. MODEL BUILDERS & TRAINING UTILITIES


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def build_group_model(num_classes: int, dropout: float = 0.3) -> nn.Module:
    """EfficientNet-B0 for group classification."""
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_sub_model(num_classes: int, dropout: float = 0.3) -> nn.Module:
    """EfficientNet-B2 for sub-class classification within a group."""
    model = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model


def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False

def unfreeze_top_blocks(model, num_blocks=3):
    total_blocks = len([n for n in model.features]) - 1
    unfreeze_from = total_blocks - num_blocks
    for name, param in model.named_parameters():
        block_match = False
        for i in range(unfreeze_from, total_blocks + 1):
            if f'features.{i}' in name:
                block_match = True
        if block_match or 'classifier' in name:
            param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return f'{trainable:,} / {total:,} ({100*trainable/total:.1f}%)'


class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val   = val
        self.sum  += val * n
        self.count += n
        self.avg   = self.sum / self.count


def train_epoch(model, loader, criterion, optimizer, scaler, num_classes):
    model.train()
    loss_m, top1_m = AverageMeter(), AverageMeter()
    pbar = tqdm(loader, desc='  Train', leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        bs = imgs.size(0)
        top1 = (outputs.argmax(1) == labels).float().mean().item()
        loss_m.update(loss.item(), bs)
        top1_m.update(top1, bs)
        pbar.set_postfix(loss=f'{loss_m.avg:.4f}', top1=f'{top1_m.avg:.3f}')
    return loss_m.avg, top1_m.avg


@torch.no_grad()
def val_epoch(model, loader, criterion, num_classes):
    """Validate model and calculate accuracy + macro/weighted Precision, Recall, F1."""
    model.eval()
    loss_m, top1_m = AverageMeter(), AverageMeter()
    all_labels = []
    all_preds = []

    pbar = tqdm(loader, desc='  Val  ', leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        preds = outputs.argmax(1)
        bs = imgs.size(0)
        top1 = (preds == labels).float().mean().item()
        loss_m.update(loss.item(), bs)
        top1_m.update(top1, bs)

        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

        pbar.set_postfix(loss=f'{loss_m.avg:.4f}', top1=f'{top1_m.avg:.3f}')

    label_ids = list(range(num_classes))
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, labels=label_ids, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, labels=label_ids, average='weighted', zero_division=0
    )

    metrics = {
        'balanced_accuracy': balanced_accuracy_score(all_labels, all_preds),
        'mcc': matthews_corrcoef(all_labels, all_preds),
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f1,
    }
    return loss_m.avg, top1_m.avg, metrics


def run_phase(phase_name, num_epochs, lr, model, train_loader, val_loader,
              criterion, scaler, num_classes, model_name='model', best_val_acc=0.0,
              save_path=None, patience=5):
    """Train one phase, return best_val_acc."""
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr * 0.01)
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        print(f'  [{phase_name}] Epoch {epoch}/{num_epochs}')
        tr_loss, tr_top1 = train_epoch(model, train_loader, criterion, optimizer, scaler, num_classes)
        vl_loss, vl_top1, val_metrics = val_epoch(model, val_loader, criterion, num_classes)
        scheduler.step()
        print(f'    Train loss={tr_loss:.4f} top1={tr_top1:.4f}')
        print(f'    Val   loss={vl_loss:.4f} top1={vl_top1:.4f}')
        print(
            '    Val   macro: '
            f'P={val_metrics["macro_precision"]:.4f} '
            f'R={val_metrics["macro_recall"]:.4f} '
            f'F1={val_metrics["macro_f1"]:.4f}'
        )
        print(
            '    Val   weighted: '
            f'P={val_metrics["weighted_precision"]:.4f} '
            f'R={val_metrics["weighted_recall"]:.4f} '
            f'F1={val_metrics["weighted_f1"]:.4f}'
        )

        # Save one row per epoch for this model.
        epoch_metric_rows.append({
            'model_name': model_name,
            'phase': phase_name,
            'epoch': epoch,
            'train_loss': tr_loss,
            'train_top1': tr_top1,
            'val_loss': vl_loss,
            'val_top1': vl_top1,
            'balanced_accuracy': val_metrics['balanced_accuracy'],
            'mcc': val_metrics['mcc'],
            'macro_precision': val_metrics['macro_precision'],
            'macro_recall': val_metrics['macro_recall'],
            'macro_f1': val_metrics['macro_f1'],
            'weighted_precision': val_metrics['weighted_precision'],
            'weighted_recall': val_metrics['weighted_recall'],
            'weighted_f1': val_metrics['weighted_f1'],
        })
        save_metrics_tables(OUTPUT_DIR)

        if vl_top1 > best_val_acc:
            best_val_acc = vl_top1
            if save_path:
                torch.save({
                    'model_state': model.state_dict(),
                    'val_acc': best_val_acc,
                    'val_metrics': val_metrics,
                }, save_path)
            print(f'    ✅ New best val_acc={best_val_acc:.4f}')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'    ⏹ Early stopping')
                break
    return best_val_acc


def train_full_pipeline(model, train_loader, val_loader, num_classes, save_path,
                        p1_epochs, p1_lr, p2_epochs, p2_lr, p3_epochs, p3_lr,
                        model_name='model'):
    """Run 3-phase training: head-only → top blocks → full fine-tune."""
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.cuda.amp.GradScaler()
    best_acc = 0.0

    # Phase 1 — head only
    freeze_backbone(model)
    print(f'  Phase 1 — Trainable: {count_trainable(model)}')
    best_acc = run_phase('P1-head', p1_epochs, p1_lr, model, train_loader, val_loader,
                         criterion, scaler, num_classes, model_name, best_acc, save_path)

    # Phase 2 — top 3 blocks
    unfreeze_top_blocks(model, num_blocks=3)
    print(f'  Phase 2 — Trainable: {count_trainable(model)}')
    best_acc = run_phase('P2-top3', p2_epochs, p2_lr, model, train_loader, val_loader,
                         criterion, scaler, num_classes, model_name, best_acc, save_path)

    # Phase 3 — full fine-tune
    unfreeze_all(model)
    print(f'  Phase 3 — Trainable: {count_trainable(model)}')
    best_acc = run_phase('P3-full', p3_epochs, p3_lr, model, train_loader, val_loader,
                         criterion, scaler, num_classes, model_name, best_acc, save_path, patience=7)

    print(f'  🏆 Best val accuracy: {best_acc*100:.2f}%')
    return best_acc


def evaluate_model_metrics(model, val_loader, num_classes, model_name='model'):
    """Evaluate a trained model and print final macro/weighted Precision, Recall, F1."""
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    val_loss, val_acc, metrics = val_epoch(model, val_loader, criterion, num_classes)
    print(f'\n  📊 FINAL METRICS — {model_name}')
    print(f'    Accuracy          : {val_acc:.4f}')
    print(f'    Balanced Accuracy : {metrics["balanced_accuracy"]:.4f}')
    print(f'    MCC               : {metrics["mcc"]:.4f}')
    print(f'    Macro Precision   : {metrics["macro_precision"]:.4f}')
    print(f'    Macro Recall      : {metrics["macro_recall"]:.4f}')
    print(f'    Macro F1          : {metrics["macro_f1"]:.4f}')
    print(f'    Weighted Precision: {metrics["weighted_precision"]:.4f}')
    print(f'    Weighted Recall   : {metrics["weighted_recall"]:.4f}')
    print(f'    Weighted F1       : {metrics["weighted_f1"]:.4f}')

    result = {'model_name': model_name, 'accuracy': val_acc, 'loss': val_loss, **metrics}
    final_metric_rows.append(result)
    save_metrics_tables(OUTPUT_DIR)

    # Return without model_name so checkpoint metadata stays compact.
    return {'accuracy': val_acc, 'loss': val_loss, **metrics}

print('Training utilities defined ✓')


## 7. TRAIN GROUP CLASSIFIER (EfficientNet-B0 → 8 groups)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. TRAIN GROUP CLASSIFIER (EfficientNet-B0 → 8 groups)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

print('='*60)
print('  TRAINING GROUP CLASSIFIER (EfficientNet-B0 → 8 groups)')
print('='*60)

group_train_tf, group_val_tf = get_transforms(GROUP_IMG_SIZE)

# Build group-level datasets
group_train_ds = GroupLabelDataset(
    TRAIN_DIR, list(class_to_group.keys()), class_to_group, group_to_idx,
    transform=group_train_tf
)

if VAL_DIR is not None:
    group_val_ds = GroupLabelDataset(
        VAL_DIR, list(class_to_group.keys()), class_to_group, group_to_idx,
        transform=group_val_tf
    )
else:
    val_size = int(0.2 * len(group_train_ds))
    train_size = len(group_train_ds) - val_size
    group_train_ds, group_val_ds = random_split(
        group_train_ds, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

group_train_loader = DataLoader(group_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=NUM_WORKERS, pin_memory=True)
group_val_loader   = DataLoader(group_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)

print(f'Group train samples: {len(group_train_ds)}')
print(f'Group val samples  : {len(group_val_ds)}')

# Build and train
group_model = build_group_model(NUM_GROUPS, DROPOUT).to(DEVICE)
GROUP_MODEL_PATH = OUTPUT_DIR / 'best_group_effb0.pth'

group_best_acc = train_full_pipeline(
    group_model, group_train_loader, group_val_loader,
    NUM_GROUPS, GROUP_MODEL_PATH,
    GROUP_PHASE1_EPOCHS, GROUP_PHASE1_LR,
    GROUP_PHASE2_EPOCHS, GROUP_PHASE2_LR,
    GROUP_PHASE3_EPOCHS, GROUP_PHASE3_LR,
    model_name='GROUP_EFFB0',
)

group_final_metrics = evaluate_model_metrics(
    group_model, group_val_loader, NUM_GROUPS, model_name='GROUP_EFFB0'
)

# Save final checkpoint with metadata
group_ckpt = {
    'model_state': group_model.state_dict(),
    'group_names': GROUP_NAMES,
    'group_to_idx': group_to_idx,
    'num_groups': NUM_GROUPS,
    'img_size': GROUP_IMG_SIZE,
    'best_val_acc': group_best_acc,
    'final_metrics': group_final_metrics,
}
torch.save(group_ckpt, OUTPUT_DIR / 'group_effb0_final.pth')
print(f'\nGroup model saved ✓')


## 8. TRAIN SUB-CLASS CLASSIFIERS (8 × EfficientNet-B2)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. TRAIN SUB-CLASS CLASSIFIERS (8 × EfficientNet-B2)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

print('='*60)
print('  TRAINING SUB-CLASS CLASSIFIERS (8 × EfficientNet-B2)')
print('='*60)

sub_train_tf, sub_val_tf = get_transforms(SUB_IMG_SIZE)
sub_results = {}

for group_name, group_classes in GROUP_CLASSES.items():
    print(f'\n{"─"*60}')
    print(f'  Group: {group_name}  ({len(group_classes)} classes)')
    print(f'{"─"*60}')

    # Build class_to_idx for this group (local indices 0..N-1)
    sorted_classes = sorted(group_classes)
    local_class_to_idx = {c: i for i, c in enumerate(sorted_classes)}
    num_local = len(sorted_classes)

    # Build datasets
    sub_train_ds = FoodDataset(TRAIN_DIR, local_class_to_idx, transform=sub_train_tf)

    if VAL_DIR is not None:
        sub_val_ds = FoodDataset(VAL_DIR, local_class_to_idx, transform=sub_val_tf)
    else:
        val_sz = int(0.2 * len(sub_train_ds))
        tr_sz = len(sub_train_ds) - val_sz
        sub_train_ds, sub_val_ds = random_split(
            sub_train_ds, [tr_sz, val_sz],
            generator=torch.Generator().manual_seed(SEED)
        )

    if len(sub_train_ds) == 0:
        print(f'  ⚠️ No training data for {group_name}, skipping!')
        continue

    sub_train_loader = DataLoader(sub_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                   num_workers=NUM_WORKERS, pin_memory=True)
    sub_val_loader   = DataLoader(sub_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

    print(f'  Train: {len(sub_train_ds)} | Val: {len(sub_val_ds)} | Classes: {num_local}')

    # Build and train model
    sub_model = build_sub_model(num_local, DROPOUT).to(DEVICE)
    save_path = OUTPUT_DIR / f'best_sub_{group_name}_effb2.pth'

    best_acc = train_full_pipeline(
        sub_model, sub_train_loader, sub_val_loader,
        num_local, save_path,
        SUB_PHASE1_EPOCHS, SUB_PHASE1_LR,
        SUB_PHASE2_EPOCHS, SUB_PHASE2_LR,
        SUB_PHASE3_EPOCHS, SUB_PHASE3_LR,
        model_name=f'SUB_{group_name}_EFFB2',
    )

    final_metrics = evaluate_model_metrics(
        sub_model, sub_val_loader, num_local, model_name=f'SUB_{group_name}_EFFB2'
    )

    # Save final with metadata
    sub_ckpt = {
        'model_state': sub_model.state_dict(),
        'group_name': group_name,
        'class_names': sorted_classes,
        'class_to_idx': local_class_to_idx,
        'num_classes': num_local,
        'img_size': SUB_IMG_SIZE,
        'best_val_acc': best_acc,
        'final_metrics': final_metrics,
    }
    torch.save(sub_ckpt, OUTPUT_DIR / f'sub_{group_name}_effb2_final.pth')

    sub_results[group_name] = {
        'num_classes': num_local,
        'best_acc': best_acc,
        'final_metrics': final_metrics,
    }

    # Free GPU memory
    del sub_model
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('  ALL SUB-CLASS MODELS TRAINED')

# ============================================================
# TEST SET EVALUATION FOR 9 MODELS
# ============================================================

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
import pandas as pd

test_metrics_records = []
confusion_output_dir = OUTPUT_DIR / 'confusion_matrices'
confusion_output_dir.mkdir(exist_ok=True)


@torch.no_grad()
def evaluate_model_on_loader(model, loader, criterion, model_name, class_names=None):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0.0
    total_samples = 0

    for imgs, labels in tqdm(loader, desc=f'Test {model_name}', leave=False):
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        preds = outputs.argmax(1)
        probs = torch.softmax(outputs, dim=1)

        bs = imgs.size(0)
        total_loss += loss.item() * bs
        total_samples += bs

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / total_samples
    acc = accuracy_score(all_labels, all_preds)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    mcc = matthews_corrcoef(all_labels, all_preds)

    if class_names is None:
        num_classes = int(max(max(all_labels), max(all_preds)) + 1) if total_samples > 0 else 0
        class_names = [str(i) for i in range(num_classes)]
    else:
        class_names = list(class_names)
        num_classes = len(class_names)

    label_ids = list(range(num_classes))
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=label_ids,
        average='macro',
        zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=label_ids,
        average='weighted',
        zero_division=0
    )

    cm = confusion_matrix(all_labels, all_preds, labels=label_ids)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    safe_model_name = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in model_name)
    report = classification_report(
        all_labels,
        all_preds,
        labels=label_ids,
        target_names=class_names,
        zero_division=0,
        output_dict=True
    )
    report_df = pd.DataFrame(report).T
    report_csv_path = confusion_output_dir / f'{safe_model_name}_classification_report.csv'
    report_df.to_csv(report_csv_path)
    cm_csv_path = confusion_output_dir / f'{safe_model_name}_confusion_matrix.csv'
    cm_png_path = confusion_output_dir / f'{safe_model_name}_confusion_matrix.png'
    cm_df.to_csv(cm_csv_path)

    y_score = np.asarray(all_probs)
    y_true_bin = label_binarize(all_labels, classes=label_ids)
    roc_csv_path = confusion_output_dir / f'{safe_model_name}_roc_curve.csv'
    roc_png_path = confusion_output_dir / f'{safe_model_name}_roc_curve.png'
    roc_rows = []
    roc_auc_values = {}

    if y_score.ndim == 2 and y_score.shape[1] == len(class_names) and len(class_names) > 1:
        try:
            roc_auc_values['roc_auc_macro_ovr'] = roc_auc_score(
                all_labels, y_score, labels=label_ids, multi_class='ovr', average='macro'
            )
            roc_auc_values['roc_auc_weighted_ovr'] = roc_auc_score(
                all_labels, y_score, labels=label_ids, multi_class='ovr', average='weighted'
            )
            roc_auc_values['roc_auc_micro_ovr'] = roc_auc_score(
                y_true_bin, y_score, average='micro'
            )
        except ValueError as exc:
            print(f'ROC AUC skipped for {model_name}: {exc}')

        plt.figure(figsize=(9, 7))
        for class_idx, class_name in enumerate(class_names):
            if len(np.unique(y_true_bin[:, class_idx])) < 2:
                continue
            fpr, tpr, _ = roc_curve(y_true_bin[:, class_idx], y_score[:, class_idx])
            class_auc = auc(fpr, tpr)
            for x, y in zip(fpr, tpr):
                roc_rows.append({'class_name': class_name, 'fpr': x, 'tpr': y, 'auc': class_auc})
            plt.plot(fpr, tpr, linewidth=1, alpha=0.55, label=f'{class_name} AUC={class_auc:.3f}')

        fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_score.ravel())
        micro_auc = auc(fpr_micro, tpr_micro)
        for x, y in zip(fpr_micro, tpr_micro):
            roc_rows.append({'class_name': 'micro_average', 'fpr': x, 'tpr': y, 'auc': micro_auc})
        plt.plot(fpr_micro, tpr_micro, color='black', linewidth=2.5, label=f'micro AUC={micro_auc:.3f}')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'One-vs-Rest ROC - {model_name}')
        plt.legend(loc='lower right', fontsize=7, ncol=2)
        plt.tight_layout()
        plt.savefig(roc_png_path, dpi=160, bbox_inches='tight')
        plt.show()
        pd.DataFrame(roc_rows).to_csv(roc_csv_path, index=False)

    print(f'\n📊 TEST METRICS — {model_name}')
    print(f'Loss               : {avg_loss:.4f}')
    print(f'Accuracy           : {acc:.4f}')
    print(f'Balanced Accuracy  : {balanced_acc:.4f}')
    print(f'MCC                : {mcc:.4f}')
    print(f'Macro Precision    : {macro_p:.4f}')
    print(f'Macro Recall       : {macro_r:.4f}')
    print(f'Macro F1           : {macro_f1:.4f}')
    print(f'Weighted Precision : {weighted_p:.4f}')
    print(f'Weighted Recall    : {weighted_r:.4f}')
    print(f'Weighted F1        : {weighted_f1:.4f}')
    print(f'Classification CSV : {report_csv_path}')
    print(f'Confusion CSV      : {cm_csv_path}')
    print(f'Confusion PNG      : {cm_png_path}')
    print(f'ROC curve CSV      : {roc_csv_path}')
    print(f'ROC curve PNG      : {roc_png_path}')
    print(f'\nCLASSIFICATION REPORT — {model_name}')
    print(classification_report(
        all_labels,
        all_preds,
        labels=label_ids,
        target_names=class_names,
        zero_division=0
    ))
    print(f'\nCONFUSION MATRIX — {model_name}')
    display(cm_df)

    fig_size = max(8, min(24, 0.45 * max(num_classes, 1)))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax,
        cmap='Blues',
        xticks_rotation=90,
        colorbar=False,
        values_format='d'
    )
    ax.set_title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.savefig(cm_png_path, dpi=160, bbox_inches='tight')
    plt.show()

    return {
        'model_name': model_name,
        'split': 'test',
        'loss': avg_loss,
        'accuracy': acc,
        'balanced_accuracy': balanced_acc,
        'mcc': mcc,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f1,
        'num_samples': total_samples,
        'classification_report_csv': str(report_csv_path),
        'confusion_matrix_csv': str(cm_csv_path),
        'confusion_matrix_png': str(cm_png_path),
        'roc_curve_csv': str(roc_csv_path),
        'roc_curve_png': str(roc_png_path),
        **roc_auc_values,
    }


criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)


# ----------------------------
# 1) Test GROUP model
# ----------------------------

_, group_test_tf = get_transforms(GROUP_IMG_SIZE)

group_test_ds = GroupLabelDataset(
    TEST_DIR,
    list(class_to_group.keys()),
    class_to_group,
    group_to_idx,
    transform=group_test_tf
)

group_test_loader = DataLoader(
    group_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

group_test_metric = evaluate_model_on_loader(
    group_model,
    group_test_loader,
    criterion,
    model_name='GROUP_EFFB0',
    class_names=[name for name, _ in sorted(group_to_idx.items(), key=lambda item: item[1])]
)

test_metrics_records.append(group_test_metric)
pd.DataFrame(test_metrics_records).to_csv(OUTPUT_DIR / 'test_metrics_summary.csv', index=False)
pd.DataFrame(test_metrics_records).to_excel(OUTPUT_DIR / 'test_metrics_summary.xlsx', index=False)
print(f'Intermediate test metrics saved after GROUP_EFFB0: {OUTPUT_DIR / "test_metrics_summary.csv"}')


# ----------------------------
# 2) Test 8 SUB models
# ----------------------------

_, sub_test_tf = get_transforms(SUB_IMG_SIZE)

for group_name, group_classes in GROUP_CLASSES.items():
    print(f'\n{"="*60}')
    print(f'Testing sub-model: {group_name}')
    print(f'{"="*60}')

    sorted_classes = sorted(group_classes)
    local_class_to_idx = {c: i for i, c in enumerate(sorted_classes)}
    num_local = len(sorted_classes)

    sub_test_ds = FoodDataset(
        TEST_DIR,
        local_class_to_idx,
        transform=sub_test_tf
    )

    if len(sub_test_ds) == 0:
        print(f'⚠️ No test data for {group_name}, skipping.')
        continue

    sub_test_loader = DataLoader(
        sub_test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    # Load best sub-model checkpoint
    sub_model = build_sub_model(num_local, DROPOUT).to(DEVICE)
    ckpt_path = OUTPUT_DIR / f'best_sub_{group_name}_effb2.pth'

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    sub_model.load_state_dict(ckpt['model_state'])

    metric = evaluate_model_on_loader(
        sub_model,
        sub_test_loader,
        criterion,
        model_name=f'SUB_{group_name}_EFFB2',
        class_names=sorted_classes
    )

    test_metrics_records.append(metric)
    pd.DataFrame(test_metrics_records).to_csv(OUTPUT_DIR / 'test_metrics_summary.csv', index=False)
    pd.DataFrame(test_metrics_records).to_excel(OUTPUT_DIR / 'test_metrics_summary.xlsx', index=False)
    print(f'Intermediate test metrics saved after SUB_{group_name}_EFFB2: {OUTPUT_DIR / "test_metrics_summary.csv"}')

    del sub_model
    torch.cuda.empty_cache()


# ----------------------------
# 3) Save test metrics
# ----------------------------

test_metrics_df = pd.DataFrame(test_metrics_records)

test_csv_path = OUTPUT_DIR / 'test_metrics_summary.csv'
test_excel_path = OUTPUT_DIR / 'test_metrics_summary.xlsx'

test_metrics_df.to_csv(test_csv_path, index=False)
test_metrics_df.to_excel(test_excel_path, index=False)

print('\n✅ Test metrics saved:')
print(test_csv_path)
print(test_excel_path)

display(test_metrics_df)

print(f'{"="*60}')
for gname, res in sub_results.items():
    m = res['final_metrics']
    print(
        f'  {gname:20s}: {res["num_classes"]:3d} classes | '
        f'best_val_acc = {res["best_acc"]*100:.2f}% | '
        f'balanced_acc = {m["balanced_accuracy"]:.4f} | '
        f'mcc = {m["mcc"]:.4f} | '
        f'macro_f1 = {m["macro_f1"]:.4f} | weighted_f1 = {m["weighted_f1"]:.4f}'
    )

# Save final CSV/Excel one more time after all 9 models are complete.
epoch_metrics_df, final_metrics_df = save_metrics_tables(OUTPUT_DIR)
print('\nFinal metrics table:')
display(final_metrics_df)


## 9. HIERARCHICAL INFERENCE PIPELINE

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. HIERARCHICAL INFERENCE PIPELINE


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def load_all_models(output_dir):
    """Load group model + all sub-class models for inference."""
    # Load group model
    group_ckpt = torch.load(output_dir / 'group_effb0_final.pth', map_location=DEVICE, weights_only=False)
    group_model = build_group_model(group_ckpt['num_groups']).to(DEVICE)
    group_model.load_state_dict(group_ckpt['model_state'])
    group_model.eval()

    # Load sub models
    sub_models = {}
    for gname in group_ckpt['group_names']:
        fpath = output_dir / f'sub_{gname}_effb2_final.pth'
        if not fpath.exists():
            print(f'Warning: missing sub-model for {gname}')
            continue
        sub_ckpt = torch.load(fpath, map_location=DEVICE, weights_only=False)
        sub_model = build_sub_model(sub_ckpt['num_classes']).to(DEVICE)
        sub_model.load_state_dict(sub_ckpt['model_state'])
        sub_model.eval()
        sub_models[gname] = {
            'model': sub_model,
            'class_names': sub_ckpt['class_names'],
        }

    return group_model, group_ckpt, sub_models


@torch.no_grad()
def predict_hierarchical(image_path, group_model, group_ckpt, sub_models):
    """
    2-stage prediction:
      Stage 1: EfficientNet-B0 → predict group
      Stage 2: EfficientNet-B2 (group-specific) → predict sub-class
    """
    img = Image.open(image_path).convert('RGB')

    # Stage 1: Group prediction
    _, group_val_tf = get_transforms(group_ckpt['img_size'])
    group_tensor = group_val_tf(img).unsqueeze(0).to(DEVICE)
    group_output = group_model(group_tensor)
    group_probs = torch.softmax(group_output, dim=1).squeeze().cpu().numpy()
    group_idx = group_probs.argmax()
    group_name = group_ckpt['group_names'][group_idx]
    group_conf = group_probs[group_idx]

    # Stage 2: Sub-class prediction
    if group_name not in sub_models:
        return group_name, group_conf, 'unknown', 0.0

    sub_info = sub_models[group_name]
    _, sub_val_tf = get_transforms(SUB_IMG_SIZE)
    sub_tensor = sub_val_tf(img).unsqueeze(0).to(DEVICE)
    sub_output = sub_info['model'](sub_tensor)
    sub_probs = torch.softmax(sub_output, dim=1).squeeze().cpu().numpy()
    sub_idx = sub_probs.argmax()
    sub_name = sub_info['class_names'][sub_idx]
    sub_conf = sub_probs[sub_idx]

    return group_name, group_conf, sub_name, sub_conf


# Load and test
group_model_inf, group_ckpt_inf, sub_models_inf = load_all_models(OUTPUT_DIR)

# Test on random image
sample_cls = random.choice(list(class_to_group.keys()))
sample_dir = (VAL_DIR or TRAIN_DIR) / sample_cls
sample_imgs = list(sample_dir.glob('*'))
if sample_imgs:
    test_img = str(random.choice(sample_imgs))
    gname, gconf, sname, sconf = predict_hierarchical(
        test_img, group_model_inf, group_ckpt_inf, sub_models_inf
    )
    print(f'\nTrue label : {sample_cls}')
    print(f'Predicted  : {gname} ({gconf:.1%}) → {sname} ({sconf:.1%})')

    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(Image.open(test_img))
    ax.set_title(f'Group: {gname} ({gconf:.0%})\nClass: {sname} ({sconf:.0%})', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# Run after group_model_inf/sub_models_inf are loaded.
hierarchical_e2e_metric, hierarchical_e2e_predictions, hierarchical_e2e_cm = evaluate_hierarchical_end_to_end(
    group_model_inf,
    group_ckpt_inf,
    sub_models_inf,
    TEST_DIR
)


## 10. SUMMARY & OUTPUT FILES

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. SUMMARY & OUTPUT FILES


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

print('='*60)
print('         TRAINING COMPLETE — ALL MODELS')
print('='*60)
print(f'\n  Group classifier (EfficientNet-B0):')
print(f'    Best val acc: {group_best_acc*100:.2f}%')
print(f'    Saved: {GROUP_MODEL_PATH.name}')
print()
print(f'  Sub-class classifiers (EfficientNet-B2):')
for gname, res in sub_results.items():
    print(f'    {gname:20s}: {res["best_acc"]*100:.2f}% ({res["num_classes"]} classes)')
print()
print(f'Output directory: {OUTPUT_DIR}')
print()
for f in sorted(OUTPUT_DIR.glob('*.pth')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<50s} {size_mb:.1f} MB')
print('='*60)


In [ ]:
# ============================================================
# TEST YOUR OWN IMAGE
# ============================================================

from PIL import Image
import matplotlib.pyplot as plt

def test_my_image(image_path):
    gname, gconf, sname, sconf = predict_hierarchical(
        image_path,
        group_model_inf,
        group_ckpt_inf,
        sub_models_inf
    )

    print('Prediction result:')
    print(f'Group     : {gname} ({gconf:.2%})')
    print(f'Food class: {sname} ({sconf:.2%})')

    img = Image.open(image_path).convert('RGB')

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(
        f'Group: {gname} ({gconf:.1%})\nClass: {sname} ({sconf:.1%})',
        fontweight='bold'
    )
    plt.axis('off')
    plt.show()

    return {
        'image_path': image_path,
        'predicted_group': gname,
        'group_confidence': gconf,
        'predicted_class': sname,
        'class_confidence': sconf,
    }

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/banh_beo.jpg')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/banh_chung.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/banh_cuon.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/cha_ca_la_vong.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/khau_nhuc.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/pho_bo.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/rau_muong_xao.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/tau_hu_non.png')
result

In [ ]:
result = test_my_image('/kaggle/input/datasets/vhutin/realistic-test-food-image/Testing_realistic_image/trung_vit_lon.png')
result